In [0]:
# ============================================
# Stage 4.7: Load FactOrderHeader (Order Grain)
# ============================================

from pyspark.sql import functions as F

raw_base = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/"
cur_base = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/"

# 源数据
df_header = spark.read.parquet(raw_base + "SalesOrderHeader/")

# 维度表
dim_customer = spark.read.format("delta").load(cur_base + "DimCustomer/")
dim_salesp   = spark.read.format("delta").load(cur_base + "DimSalesPerson/")
dim_terr     = spark.read.format("delta").load(cur_base + "DimTerritory/")

print(f"SalesOrderHeader: {df_header.count()}")

SalesOrderHeader: 31465


In [0]:
# JOIN + 代理键查找
fact_header = (df_header.alias("h")
    .join(dim_customer.alias("dc"), 
          (F.col("h.CustomerID") == F.col("dc.CustomerID")) & (F.col("dc.is_current") == 1), 
          how="left")
    .join(dim_salesp.alias("dsp"), 
          (F.col("h.SalesPersonID") == F.col("dsp.SalesPersonID")) & (F.col("dsp.is_current") == 1), 
          how="left")
    .join(dim_terr.alias("dt"), 
          F.col("h.TerritoryID") == F.col("dt.TerritoryID"), 
          how="left")
    .select(
        F.date_format(F.col("h.OrderDate"), "yyyyMMdd").cast("int").alias("DateKey"),
        F.col("dc.CustomerKey").alias("CustomerKey"),
        F.col("dsp.SalesPersonKey").alias("SalesPersonKey"),
        F.col("dt.TerritoryKey").alias("TerritoryKey"),
        F.col("h.SalesOrderID").alias("SalesOrderID"),
        F.col("h.TaxAmt").cast("decimal(19,4)").alias("TaxAmt"),
        F.col("h.Freight").cast("decimal(19,4)").alias("Freight"),
        F.col("h.TotalDue").cast("decimal(19,4)").alias("TotalDue")
    )
)

print(f"FactOrderHeader rows: {fact_header.count()}")
# 期望：31,465

FactOrderHeader rows: 31465


In [0]:
# 写入 Curated Zone
curated_path = cur_base + "FactOrderHeader/"

fact_header.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ FactOrderHeader written to {curated_path}")

✅ FactOrderHeader written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/FactOrderHeader/


In [0]:
# 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
# 期望：31465

# 检查代理键 NULL
df_check.select(
    F.sum(F.when(F.col("DateKey").isNull(), 1).otherwise(0)).alias("NullDateKey"),
    F.sum(F.when(F.col("CustomerKey").isNull(), 1).otherwise(0)).alias("NullCustomerKey"),
    F.sum(F.when(F.col("SalesPersonKey").isNull(), 1).otherwise(0)).alias("NullSalesPersonKey"),
    F.sum(F.when(F.col("TerritoryKey").isNull(), 1).otherwise(0)).alias("NullTerritoryKey")
).show()
# 期望：SalesPersonKey = 27659，其他 = 0

# 金额对账
df_check.select(
    F.sum("TaxAmt").alias("TotalTax"),
    F.sum("Freight").alias("TotalFreight"),
    F.sum("TotalDue").alias("TotalDue")
).show()
# 应该和 OLTP 一致

Total rows: 31465
+-----------+---------------+------------------+----------------+
|NullDateKey|NullCustomerKey|NullSalesPersonKey|NullTerritoryKey|
+-----------+---------------+------------------+----------------+
|          0|              0|             27659|               0|
+-----------+---------------+------------------+----------------+

+-------------+------------+--------------+
|     TotalTax|TotalFreight|      TotalDue|
+-------------+------------+--------------+
|10186974.4602|3183430.2518|123216786.1159|
+-------------+------------+--------------+

